<a href="https://colab.research.google.com/github/ubsuny/PHY386/blob/Homework2026/2026/handson/TimeSeriesPredictionML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Time Series Prediction in Physics with Machine Learning

> **A note before we start.** Predicting what comes next from past data is one of the oldest activities in physics, not an "AI thing". Kepler extrapolated planetary positions from Tycho Brahe's tables; ship navigators extrapolate tides; meteorologists fit equations to weather charts. Whenever you fit a line through data points and read off a value just past the last one, you are doing time-series prediction. What changes when we say "machine learning" is only the *family of functions* we fit — not the goal.

## Learning Outcomes
- Reframe a time series as a **supervised learning** problem using lag features (a sliding window of past values predicts the next value).
- Train and compare three regression models — Linear Regression, Random Forest Regressor, and Multi-Layer Perceptron — on physical data.
- Distinguish **one-step-ahead** prediction (cheap, easy) from **multi-step rollout** (the model is fed its own predictions — errors compound).
- Recognize that *good one-step accuracy does not imply long-term forecasting power*, especially for chaotic systems.

## Three physics examples
1. **Damped pendulum** — clean, periodic, low noise. The easy case.
2. **Sunspot number** (real SILSO monthly data, 1749–present) — quasi-periodic, noisy, real measurements.
3. **Lorenz attractor** — deterministic chaos. Short-term predictable, long-term not.

## Background: Time series as supervised learning

A time series is a sequence $\{x_t\}_{t=0}^{N-1}$. We want to predict $x_{t+1}$ from the recent past. The standard trick is to slide a **window** of length $L$ over the series and turn it into an ordinary regression dataset:

$$
\underbrace{(x_{t-L+1}, x_{t-L+2}, \dots, x_t)}_{\text{features (input)}} \;\longrightarrow\; \underbrace{x_{t+1}}_{\text{target}}
$$

Once in this form, *any* regressor works. The window length $L$ is a hyperparameter — typically a few times the dominant timescale of the system.

**One-step prediction:** feed the model the true past, predict one step ahead. Errors do not accumulate.

**Rollout (free-running) prediction:** feed the model its *own* prediction back as input, autoregressively. Errors compound. This is the hard test.

In [ ]:
# Required packages (uncomment for Colab or fresh environment):
# %pip install numpy pandas matplotlib scikit-learn scipy --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import odeint

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

plt.rcParams.update({
    'font.size': 11, 'axes.labelsize': 11, 'axes.titlesize': 11,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
    'xtick.direction': 'in', 'ytick.direction': 'in',
    'xtick.top': True, 'ytick.right': True,
})

rng = np.random.default_rng(42)

## Helper functions

Two utilities we reuse throughout the notebook:
- `make_windows` reshapes a 1-D series into the (features, target) matrix described above.
- `rollout` runs a trained regressor autoregressively for `n_steps` into the future.

In [ ]:
def make_windows(series: np.ndarray, window: int) -> tuple[np.ndarray, np.ndarray]:
    """Convert a 1-D series to (X, y) for one-step-ahead supervised learning.

    Parameters
    ----------
    series : np.ndarray, shape (N,)
        The time series.
    window : int
        Number of past samples used as features.

    Returns
    -------
    X : np.ndarray, shape (N - window, window)
        Each row is a length-`window` slice of past samples.
    y : np.ndarray, shape (N - window,)
        The next sample after each window.
    """
    X = np.lib.stride_tricks.sliding_window_view(series, window)[:-1]
    y = series[window:]
    return X, y


def rollout(model, seed: np.ndarray, n_steps: int, scaler: StandardScaler | None = None) -> np.ndarray:
    """Run a trained one-step regressor autoregressively for n_steps.

    Parameters
    ----------
    model : fitted sklearn regressor with .predict()
    seed : np.ndarray, shape (window,)
        Initial window of true values.
    n_steps : int
        Number of future steps to roll out.
    scaler : StandardScaler, optional
        If the model was trained on scaled features, pass the fitted scaler.

    Returns
    -------
    np.ndarray, shape (n_steps,)
        The predicted future values.
    """
    window = seed.copy()
    out = np.empty(n_steps)
    for i in range(n_steps):
        x = window.reshape(1, -1)
        if scaler is not None:
            x = scaler.transform(x)
        pred = model.predict(x)[0]
        out[i] = pred
        window = np.roll(window, -1)
        window[-1] = pred
    return out

### How the helpers work — toy demonstrations

Before turning these helpers loose on noisy physics data, let's see what they do on a trivial signal: a clean sine wave $x(t) = \sin(t)$ sampled every 0.5 s. The dynamics are simple enough to verify by eye.

In [ ]:
# --- Toy 1: visualise make_windows ---
toy_t = np.arange(0, 6.0, 0.5)
toy_x = np.sin(toy_t)
toy_window = 4

X_toy, y_toy = make_windows(toy_x, toy_window)
print(f"series length: {len(toy_x)},  window: {toy_window}")
print(f"X shape: {X_toy.shape},  y shape: {y_toy.shape}")
print("\nFirst three (input window) -> target rows:")
for i in range(3):
    print(f"  X[{i}] = {np.round(X_toy[i], 2)}  ->  y[{i}] = {y_toy[i]:+.2f}")

# Show the first three windows sliding over the series
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(toy_t, toy_x, 'o-', color='lightgray', lw=1.0, label='full series')
colors = ['C0', 'C1', 'C2']
for i, c in enumerate(colors):
    win_t = toy_t[i:i + toy_window]
    win_x = toy_x[i:i + toy_window]
    ax.plot(win_t, win_x, 'o-', color=c, lw=2.0, label=f'window {i} (input)')
    ax.plot(toy_t[i + toy_window], toy_x[i + toy_window],
            marker='*', markersize=14, color=c, label=f'target y[{i}]')
ax.set_xlabel('time (s)'); ax.set_ylabel('x(t)')
ax.set_title(f'make_windows: each row = {toy_window} past samples → next sample')
ax.legend(loc='lower left', ncol=2); fig.tight_layout()

In [ ]:
# --- Toy 2: visualise rollout on the same sine wave ---
# Train a Linear Regression on the first 40 windowed samples, then roll out from a seed.
demo_t = np.arange(0, 50.0, 0.5)
demo_x = np.sin(demo_t)

W = 4
X_demo, y_demo = make_windows(demo_x, W)
demo_model = LinearRegression().fit(X_demo[:40], y_demo[:40])

seed_start = 40
seed = demo_x[seed_start:seed_start + W]            # last W true points before forecast starts
n_steps = 30
pred = rollout(demo_model, seed, n_steps)          # autoregressive forecast

t_seed = demo_t[seed_start:seed_start + W]
t_pred = demo_t[seed_start + W : seed_start + W + n_steps]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(demo_t, demo_x, 'o-', color='lightgray', lw=1.0, label='true series')
ax.plot(t_seed, seed, 'o-', color='C0', lw=2.0, label=f'seed (last {W} true samples)')
ax.plot(t_pred, pred, 's--', color='C3', lw=1.5, label='rollout prediction')
ax.axvline(t_pred[0] - 0.25, color='k', ls=':', lw=0.8)
ax.text(t_pred[0] + 0.2, 1.05, 'forecast region →', fontsize=9)
ax.set_xlabel('time (s)'); ax.set_ylabel('x(t)')
ax.set_title('rollout: model is fed its own predictions, autoregressively')
ax.legend(loc='lower left'); fig.tight_layout()

**What you should see:**

- The first plot shows three coloured input windows of length 4 sliding one sample at a time, each with a star marking its target. That is exactly one row of `X` and one entry of `y`.
- The second plot shows the rollout: only the **first 4** samples of the forecast region come from the truth (the seed); everything after is the model predicting from its own previous outputs. On a clean sine, a linear model with a 4-sample window is enough to track the wave indefinitely — but we will see in the chaos example that this is not always the case.

### How big should the window be?

The window length $L$ is the most important hyperparameter. Two competing pressures:

- **Too small.** The model can't *see* enough of the past to tell where in the cycle it is. A window of 2 samples on a sine wave can't even distinguish the rising part from the falling part — the model has no way to know which way the next step should go.
- **Too big.** Most of the window is ancient history that's no longer relevant to the next step. You're also adding noisy features (more numbers in each row) without adding information, so the model has more knobs to overfit and fewer rows of training data per knob (because every long window costs samples at the start of the series).

A useful rule of thumb: **the window should cover roughly one to a few characteristic timescales of the system** — one period for an oscillator, one cycle for a quasi-periodic signal, a few correlation times for a noisy process.

Let's see this on a noisy sine with period $T = 2\pi \approx 6.28$ s sampled every 0.1 s, so one period $\approx 63$ samples.

In [ ]:
# Sweep the window length on a noisy sine
ws_t = np.arange(0, 60.0, 0.1)
ws_x = np.sin(ws_t) + 0.1 * rng.standard_normal(ws_t.size)
period_samples = int(2 * np.pi / 0.1)              # ~63 samples per period

window_sizes = [2, 4, 8, 16, 32, 64, 128, 256]
rmse_per_window = []
for W in window_sizes:
    X, y = make_windows(ws_x, W)
    s = int(0.7 * len(X))
    sc = StandardScaler().fit(X[:s])
    m = LinearRegression().fit(sc.transform(X[:s]), y[:s])
    rmse = np.sqrt(mean_squared_error(y[s:], m.predict(sc.transform(X[s:]))))
    rmse_per_window.append(rmse)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(window_sizes, rmse_per_window, 'o-', color='C0', lw=1.5)
ax.axvline(period_samples, color='C3', ls='--', lw=1.0, label=f'one full period ({period_samples} samples)')
ax.set_xscale('log', base=2)
ax.set_xlabel('window length L (samples)'); ax.set_ylabel('test RMSE')
ax.set_title('Window too small → can\'t see the wave.   Window too big → overfits noise.')
ax.legend(); fig.tight_layout()

**What you should see.** A clear U-shape. Error is high for $L = 2$ — the model literally cannot tell the wave from one rising sample to the next. Error drops as $L$ approaches one period (~63 samples) and reaches its minimum somewhere between half a period and a few periods. Beyond that, error climbs again as ancient history is added without bringing new information.

The take-away: **pick the window so it spans the slowest pattern you care about**, then stop. For the pendulum we used $L = 25$ at $dt = 0.02$ s ≈ half a period; for the sunspot record we used $L = 60$ months ≈ half the 11-year cycle; for Lorenz we used $L = 30$ at $dt = 0.01$ ≈ a typical loop time on the attractor. None of these are magic numbers — they're just the natural timescale of each system, expressed in samples.

**The same idea, but visually.** Let's *watch* what each window size does. We'll train Linear Regression at three different window sizes — too small ($L=3$), about right ($L=64 \approx$ one period), and too big ($L=256 \approx$ four periods) — and then run a free rollout from the same starting point. The right plot is worth a thousand RMSE numbers.

In [ ]:
# Train Linear Regression at three window sizes and roll each forward from the same point
split_at = 400                      # all rollouts start here
n_future = 150                      # forecast 150 samples (~2.4 periods)
demo_windows = [3, 64, 256]
labels = ['too small (L=3)',
          f'about right (L=64 ≈ one period)',
          'too big (L=256 ≈ 4 periods)']

fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True, sharey=True)
for ax, W, label in zip(axes, demo_windows, labels):
    X, y = make_windows(ws_x, W)
    s = split_at - W
    sc = StandardScaler().fit(X[:s])
    m = LinearRegression().fit(sc.transform(X[:s]), y[:s])

    seed = ws_x[split_at - W : split_at]
    pred = rollout(m, seed, n_future, scaler=sc)
    truth = ws_x[split_at : split_at + n_future]
    rmse = np.sqrt(np.mean((pred - truth) ** 2))

    t_pred = ws_t[split_at : split_at + n_future]
    ax.plot(ws_t, ws_x, color='lightgray', lw=0.6, label='full noisy series')
    ax.plot(t_pred, truth, color='k', lw=1.0, label='truth (forecast region)')
    ax.plot(t_pred, pred,  color='C3', lw=1.2, label='rollout prediction')
    ax.axvline(ws_t[split_at], color='k', ls=':', lw=0.8)
    ax.set_ylabel('x(t)')
    ax.set_title(f'{label}   —   rollout RMSE = {rmse:.2f}')
    ax.legend(loc='upper right', fontsize=9)

axes[-1].set_xlabel('time (s)')
fig.tight_layout()

**Read the three panels.**

- **Top ($L=3$, too small).** The model has only three samples of recent past — barely a quarter-period — and quickly collapses to a near-constant line. With so little context it can't tell where in the cycle it is, so its safest guess is roughly the average of the noisy data.
- **Middle ($L=64$, about right).** One full period of context is plenty to pin down both phase and amplitude. The rollout tracks the true sine for the entire forecast region.
- **Bottom ($L=256$, too big).** Four periods of context is *more* information, but the model now has 256 weights to fit and a smaller training set after the longer window cuts off the start. The phase is right but the rollout is noisier — the extra weights have absorbed extra noise.

That's the real cost of choosing the wrong window size: too small and the model can't see the pattern at all; too big and you trade signal for noise.

### Why feature scaling matters

You used `StandardScaler` in HW5 and HW6 — let's see *why* it mattered. The clearest case is when your input features have **wildly different units**.

Imagine predicting some quantity $y$ from two physical measurements:

- **Pressure** in pascals — values are around $10^5$ Pa, fluctuating by maybe $\pm 1000$ Pa.
- **Temperature** in kelvin — values are around $300$ K, fluctuating by maybe $\pm 5$ K.

Suppose, secretly, that $y$ depends *only on the temperature fluctuations*. The pressure number is a huge irrelevant offset that wobbles around a big mean.

Two different ML models react very differently to this:

- **Distance-based models** (K-Nearest-Neighbours, k-means, support-vector machines, anything that uses Euclidean distance between samples) will compute $\sqrt{(\Delta P)^2 + (\Delta T)^2}$. The pressure fluctuation is $\sim 1000$ and temperature fluctuation is $\sim 5$ — the distance is essentially equal to $|\Delta P|$ and the temperature is invisible.
- **Linear models** can *technically* compensate by absorbing the scale into the coefficients, but the resulting numbers (~$10^{-5}$ for pressure, ~$1$ for temperature) are unreadable, and the optimizer's gradient steps are poorly conditioned (one direction is vastly steeper than the other).

`StandardScaler` puts both features on the same footing — zero mean, unit variance — so each gets an equal vote and the coefficients become directly comparable.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

# Two physical features: pressure in Pa and temperature in K
N = 400
delta_P = rng.normal(0, 1000, N)              # ±1000 Pa fluctuations around 10^5 Pa
delta_T = rng.normal(0,    5, N)              # ±5 K fluctuations around 300 K
P = 1e5 + delta_P
T = 300 + delta_T

# y depends ONLY on temperature deviation (small noise added)
y_true = 2.0 * delta_T + 0.1 * rng.standard_normal(N)
X      = np.column_stack([P, T])

# Inspect the raw scales
print(f"pressure    : mean = {P.mean():.1f} Pa,   std = {P.std():.1f} Pa")
print(f"temperature : mean = {T.mean():.2f} K,    std = {T.std():.2f} K")
print(f"y           : mean = {y_true.mean():+.2f},      std = {y_true.std():.2f}")
print(f"\nratio of feature std-devs (pressure / temperature) = {P.std()/T.std():.0f}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors

X_tr, X_te, y_tr, y_te = train_test_split(X, y_true, test_size=0.3, random_state=0)

# Scaler fit on TRAIN ONLY (HW5/HW6 rule — applies here too)
scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_te_s = scaler.transform(X_te)

# Pick a query point in the test set and find its 5 nearest training neighbours
# under raw vs. scaled distance.
q_idx = 0
q     = X_te[q_idx]
q_s   = X_te_s[q_idx]
q_y   = y_te[q_idx]

nn_raw = NearestNeighbors(n_neighbors=5).fit(X_tr)
nn_scl = NearestNeighbors(n_neighbors=5).fit(X_tr_s)
idx_raw = nn_raw.kneighbors(q.reshape(1, -1))[1][0]
idx_scl = nn_scl.kneighbors(q_s.reshape(1, -1))[1][0]

# What does KNN actually predict for this query? Mean of the 5 neighbour y-values.
pred_raw = y_tr[idx_raw].mean()
pred_scl = y_tr[idx_scl].mean()
print(f"query point true y       = {q_y:+.2f}")
print(f"prediction, raw-distance = {pred_raw:+.2f}   (5 neighbours' y's: {np.round(y_tr[idx_raw],1).tolist()})")
print(f"prediction, scaled       = {pred_scl:+.2f}   (5 neighbours' y's: {np.round(y_tr[idx_scl],1).tolist()})")

In [ ]:
# Visualise the failure: where are the 5 nearest neighbours actually located?
# Left:  raw axes — neighbours sit at almost exactly the query's pressure but span a wide
#                   range of temperatures (because Δpressure dominates Euclidean distance).
# Right: scaled axes — neighbours form a tight cluster around the query.

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# ----- Left panel: raw feature plane (pressure, temperature) -----
ax = axes[0]
sc = ax.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, cmap='RdBu_r',
                s=18, alpha=0.6, edgecolor='none')
ax.scatter(X_tr[idx_raw, 0], X_tr[idx_raw, 1],
           s=140, facecolor='none', edgecolor='C2', lw=2.0,
           label='5 nearest neighbours (raw distance)')
ax.scatter(*q, s=200, marker='*', color='k', zorder=5, label='query point')
ax.set_xlabel('pressure (Pa)'); ax.set_ylabel('temperature (K)')
ax.set_title('Raw axes — distance dominated by pressure\n'
             'Neighbours match query in pressure but span all temperatures')
ax.legend(loc='upper left', fontsize=9)
plt.colorbar(sc, ax=ax, label='true y')

# ----- Right panel: scaled feature plane -----
ax = axes[1]
sc = ax.scatter(X_tr_s[:, 0], X_tr_s[:, 1], c=y_tr, cmap='RdBu_r',
                s=18, alpha=0.6, edgecolor='none')
ax.scatter(X_tr_s[idx_scl, 0], X_tr_s[idx_scl, 1],
           s=140, facecolor='none', edgecolor='C2', lw=2.0,
           label='5 nearest neighbours (scaled distance)')
ax.scatter(*q_s, s=200, marker='*', color='k', zorder=5, label='query point')
ax.set_xlabel('pressure (z-scored)'); ax.set_ylabel('temperature (z-scored)')
ax.set_title('After scaling — both axes contribute equally\n'
             'Neighbours form a tight cluster around the query')
ax.set_aspect('equal')
ax.legend(loc='upper left', fontsize=9)
plt.colorbar(sc, ax=ax, label='true y')

fig.tight_layout()

**Read the two panels.**

The colour of every dot is its true value of $y$, which (by construction) depends *only on temperature*. So the colour stripes the temperature axis: cold = blue, warm = red.

- **Left (raw axes).** Pressure ranges over $\sim 5000$ Pa; temperature over $\sim 25$ K. To Euclidean distance these are just *numbers*: a 1-Pa step "costs" the same as a 1-K step. Since $\Delta P$ values are about 200× larger than $\Delta T$ values, distance is essentially $|\Delta P|$ alone. The 5 nearest neighbours therefore match the query in pressure but are scattered across **a wide range of temperatures** — and so across a wide range of colours. Averaging their y-values gives a useless prediction.
- **Right (scaled axes).** Both features are z-scored, so 1 unit of pressure-deviation and 1 unit of temperature-deviation have the same geometric length. Neighbours form a **tight cluster** around the query, all of similar colour. KNN now averages five y-values that are actually similar to the truth.

The text output above makes this concrete: the raw-distance neighbours' y-values span a much wider range than the scaled-distance neighbours, and only the scaled prediction lands close to the query's true y.

**This is also why scaling matters for gradient-trained models** like the Multi-Layer Perceptron. Imagine the loss surface in (pressure-weight, temperature-weight) space: because pressure values are $\sim 1000\times$ larger, gradient steps along the pressure-weight axis are $10^6\times$ steeper than along the temperature-weight axis. The optimizer either overshoots in the steep direction or crawls in the flat one. Z-scoring turns the loss surface into something close to a sphere, and gradient descent works the way the textbooks say it should.

**Two rules of thumb you'll want for time series too.**

1. **Scale features whose physical units differ.** Essential for distance-based models (KNN, k-means, support-vector machines) and gradient-trained networks.
2. **Fit the scaler on training data only.** This is the rule from HW5/HW6, and it matters even more for time series, where train and test are *different time intervals* of the same record. Fitting the scaler on the whole series would secretly let the model see the future's mean and variance — a leak.

The same idea applies to other preprocessing that uses statistics: **principal-component analysis, imputation medians, baseline subtraction — all of them must be fit on training data only.**

---
# Example 1 — Damped Pendulum

The equation of motion for a damped pendulum (small-angle limit) is

$$
\ddot\theta + 2\gamma\,\dot\theta + \omega_0^2\,\theta = 0,
$$

with analytic solution (under-damped, $\gamma<\omega_0$):

$$
\theta(t) = A\,e^{-\gamma t}\cos(\omega_d t + \phi),\qquad \omega_d = \sqrt{\omega_0^2 - \gamma^2}.
$$

We will:
1. Generate a clean signal, add small Gaussian noise to mimic a real measurement.
2. Train Linear Regression and Multi-Layer Perceptron to predict $\theta_{t+1}$ from the last $L$ samples.
3. Compare one-step error to free rollout.

In [ ]:
# Generate the pendulum signal
omega0 = 2.0 * np.pi      # natural angular frequency (rad/s) — period 1 s
gamma  = 0.05             # damping rate (1/s)
A, phi = 1.0, 0.0

dt = 0.02
t  = np.arange(0, 30.0, dt)
omega_d = np.sqrt(omega0**2 - gamma**2)
theta_clean = A * np.exp(-gamma * t) * np.cos(omega_d * t + phi)
theta = theta_clean + rng.normal(0, 0.01, size=t.size)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(t, theta, lw=0.8, label='measured (with noise)')
ax.plot(t, theta_clean, lw=1.2, color='C3', label='analytic')
ax.set_xlabel('time (s)'); ax.set_ylabel(r'$\theta$ (rad)')
ax.set_title('Damped pendulum')
ax.legend(); fig.tight_layout()

In [ ]:
# Build the supervised dataset and split chronologically (NEVER shuffle a time series)
WINDOW = 25                      # ~½ period at dt=0.02
X, y = make_windows(theta, WINDOW)
split = int(0.7 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"train: {X_train.shape},  test: {X_test.shape}")

In [ ]:
# Train two models
linreg = LinearRegression().fit(X_train_s, y_train)
mlp    = MLPRegressor(hidden_layer_sizes=(32, 32), max_iter=2000,
                     random_state=0).fit(X_train_s, y_train)

for name, m in [('Linear Regression', linreg), ('Multi-Layer Perceptron', mlp)]:
    yhat = m.predict(X_test_s)
    rmse = np.sqrt(mean_squared_error(y_test, yhat))
    print(f"{name:25s} one-step RMSE = {rmse:.4f}   R² = {r2_score(y_test, yhat):.4f}")

In [ ]:
# Multi-step rollout from the start of the test set
seed = theta[split:split + WINDOW]
n_future = len(y_test)

roll_lin = rollout(linreg, seed, n_future, scaler=scaler)
roll_mlp = rollout(mlp,    seed, n_future, scaler=scaler)

print(f"Linear Regression       rollout RMSE = {np.sqrt(mean_squared_error(y_test, roll_lin)):.4f}")
print(f"Multi-Layer Perceptron  rollout RMSE = {np.sqrt(mean_squared_error(y_test, roll_mlp)):.4e}")

t_test = t[WINDOW + split : WINDOW + split + n_future]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(t_test, y_test,    color='k',  lw=1.0, label='truth')
axes[0].plot(t_test, roll_lin,  color='C0', lw=1.0, label='Linear Regression rollout')
axes[0].plot(t_test, roll_mlp,  color='C1', lw=1.0, label='Multi-Layer Perceptron rollout')
axes[0].set_ylim(-1.2, 1.2)        # MLP often diverges off-scale; clip to truth range
axes[0].set_xlabel('time (s)'); axes[0].set_ylabel(r'$\theta$ (rad)')
axes[0].set_title('Rollout (clipped to truth scale)')
axes[0].legend(loc='upper right')

# Same data on a symmetric-log y-axis to show the divergence honestly
axes[1].plot(t_test, y_test,    color='k',  lw=1.0, label='truth')
axes[1].plot(t_test, roll_lin,  color='C0', lw=1.0, label='Linear Regression')
axes[1].plot(t_test, roll_mlp,  color='C1', lw=1.0, label='Multi-Layer Perceptron')
axes[1].set_yscale('symlog', linthresh=1e-2)
axes[1].set_xlabel('time (s)'); axes[1].set_ylabel(r'$\theta$ (symlog)')
axes[1].set_title('Same plot, symlog axis — MLP rollout blows up')
axes[1].legend(loc='upper left')
fig.tight_layout()

**Discussion.** One-step accuracy is comparable for the two models, but the **rollout tells a very different story**. The damped sinusoid is the sum of two complex exponentials, so a *length-$L$ linear combination* of the past samples is enough to reproduce the next sample exactly — Linear Regression rolls out for the full test segment with RMSE essentially equal to the measurement noise.

The Multi-Layer Perceptron, despite a comparable one-step error, **diverges catastrophically** under rollout — its predictions blow up by orders of magnitude after a few hundred steps. Why? Tiny biases that are invisible at one step are fed back as input at the next step, and a non-linear network has no reason to suppress them. The lesson is sharp: **good one-step error does not imply a stable rollout**, and the simpler model is the right tool when the underlying dynamics actually are linear in the lag features.

---
# Example 2 — Sunspot Number (Real Data)

The **monthly mean total sunspot number** is recorded by the [SILSO](https://www.sidc.be/SILSO/datafiles) (Sunspot Index and Long-term Solar Observations) team in Brussels and goes back to **1749**. It shows the famous ~11-year solar cycle but is far from a clean sinusoid: cycle amplitudes vary, and there are noisy month-to-month fluctuations.

We try to predict next month's sunspot number from the last 5 years of monthly values.

In [ ]:
# Pull SILSO monthly mean total sunspot number (column-formatted CSV)
URL = 'https://www.sidc.be/SILSO/INFO/snmtotcsv.php'
cols = ['year', 'month', 'frac_year', 'sn', 'sn_std', 'n_obs', 'definitive']
ssn = pd.read_csv(URL, sep=';', header=None, names=cols)
ssn = ssn[ssn['sn'] >= 0].reset_index(drop=True)   # SILSO uses -1 for missing
print(f"{len(ssn)} months from {ssn['frac_year'].min():.2f} to {ssn['frac_year'].max():.2f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(ssn['frac_year'], ssn['sn'], lw=0.6)
ax.set_xlabel('year'); ax.set_ylabel('monthly sunspot number')
ax.set_title('SILSO monthly mean total sunspot number')
fig.tight_layout()

In [ ]:
series = ssn['sn'].to_numpy(dtype=float)
WINDOW = 60                        # 5 years of monthly samples
X, y = make_windows(series, WINDOW)

# Hold out the last 30 years as the test set
split = len(X) - 12 * 30
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"train: {X_train.shape},  test (last 30 yr): {X_test.shape}")

In [ ]:
linreg = LinearRegression().fit(X_train_s, y_train)
rf     = RandomForestRegressor(n_estimators=300, max_depth=12,
                              random_state=0, n_jobs=-1).fit(X_train_s, y_train)
mlp    = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=3000,
                     random_state=0).fit(X_train_s, y_train)

# Two cheap baselines: "persistence" (predict last month) and "train-mean"
yhat_persistence = X_test[:, -1]                         # last value in each window
yhat_mean        = np.full_like(y_test, y_train.mean())

results = {}
print(f"{'model':30s}  RMSE     R²")
print('-' * 50)
for name, m in [('Linear Regression', linreg),
                ('Random Forest',     rf),
                ('Multi-Layer Perceptron', mlp)]:
    yhat = m.predict(X_test_s)
    results[name] = yhat
    rmse = np.sqrt(mean_squared_error(y_test, yhat))
    print(f"{name:30s}  {rmse:5.2f}   {r2_score(y_test, yhat):+.3f}")

print('-' * 50)
print(f"{'persistence (last month)':30s}  {np.sqrt(mean_squared_error(y_test, yhat_persistence)):5.2f}"
      f"   {r2_score(y_test, yhat_persistence):+.3f}")
print(f"{'train mean':30s}  {np.sqrt(mean_squared_error(y_test, yhat_mean)):5.2f}"
      f"   {r2_score(y_test, yhat_mean):+.3f}")

In [ ]:
# Plot one-step predictions on the held-out 30 years
t_test = ssn['frac_year'].to_numpy()[WINDOW + split : WINDOW + split + len(y_test)]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(t_test, y_test, color='k', lw=1.0, label='truth')
for name, yhat in results.items():
    ax.plot(t_test, yhat, lw=0.9, alpha=0.8, label=name)
ax.set_xlabel('year'); ax.set_ylabel('sunspot number')
ax.set_title('Sunspot number: one-step-ahead predictions (test set)')
ax.legend(); fig.tight_layout()

In [ ]:
# Long-horizon rollout — try to predict the most recent ~25 years from a single seed
n_future = 12 * 25
seed = series[split + WINDOW - WINDOW : split + WINDOW]   # first window of test set

roll_rf  = rollout(rf,  seed, n_future, scaler=scaler)
roll_mlp = rollout(mlp, seed, n_future, scaler=scaler)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(t_test[:n_future], y_test[:n_future],  color='k',  lw=1.0, label='truth')
ax.plot(t_test[:n_future], roll_rf,            color='C2', lw=0.9, label='Random Forest rollout')
ax.plot(t_test[:n_future], roll_mlp,           color='C1', lw=0.9, label='Multi-Layer Perceptron rollout')
ax.set_xlabel('year'); ax.set_ylabel('sunspot number')
ax.set_title('Sunspot number: 25-year free rollout')
ax.legend(); fig.tight_layout()

**Discussion — what the numbers really mean.**

The most important number in the table is the **persistence baseline**: "next month's sunspot count = this month's sunspot count". With *zero* model and *zero* training, that simple rule already gets RMSE ≈ 22 (R² ≈ 0.86).

Now look at the trained models — they get RMSE ≈ 19–23. That's only a few percent better than persistence. So most of what the models look like they're "learning" is just the obvious fact that *the sunspot number doesn't change much from one month to the next*. None of them is really capturing the 11-year cycle in any deep way. **Always plot a baseline.** It tells you whether your fancy model is doing real work or just inheriting credit from autocorrelation.

With the baseline in mind, the model ranking makes sense:

- **Linear Regression wins** because predicting next month from a weighted average of the last 60 months is almost exactly the right shape of model for this data. The dominant pattern is "next month is a smooth function of recent months", and a linear weighted sum captures that with no fuss.
- **Random Forest ties** Linear Regression because it can also approximate "next month ≈ weighted recent past", and there isn't enough non-linearity in the signal for its extra flexibility to help.
- **Multi-Layer Perceptron is worst** — it barely beats persistence. ~3000 monthly samples is not much data for a neural network with thousands of parameters. The network has to *discover* that the answer is mostly a linear weighted sum of recent months; Linear Regression assumes that for free. The extra flexibility of the network costs more (in bad fits to the small dataset) than it gains.

A good rule of thumb: **match the model to the data.** Lots of data and complicated dynamics? Reach for a neural network. Modest data and roughly linear dynamics? A simple linear model is often unbeatable.

The long rollouts decay toward the long-term mean because *nothing in the lag features encodes the magnetohydrodynamic mechanism that drives the cycle*. Once the model is fed its own predictions, it has no way to sustain the 11-year oscillation — it falls back to "things are usually around average". **Naive ML can interpolate a quasi-periodic record but should not be confused with a physical solar-dynamo model.**

---
# Example 3 — Lorenz Attractor (Deterministic Chaos)

Edward Lorenz's 1963 toy model of atmospheric convection:

$$
\dot x = \sigma(y-x),\quad \dot y = x(\rho-z) - y,\quad \dot z = xy - \beta z.
$$

With $\sigma=10$, $\rho=28$, $\beta=8/3$, trajectories on the attractor are **chaotic**.

### What does "chaotic" mean — and what is a Lyapunov time?

Take two starting points that are almost identical — say, differing by $10^{-8}$ in $x$ — and integrate them forward in time. In a non-chaotic system (a pendulum, a planet's orbit) the two trajectories stay close forever: a small initial error stays a small error.

In a **chaotic** system the gap between the two trajectories grows *exponentially*:

$$
\| \delta(t) \| \;\sim\; \| \delta_0 \|\, e^{\lambda_1 t}.
$$

The number $\lambda_1$ is the **largest Lyapunov exponent**. Its inverse, $1/\lambda_1$, is the **Lyapunov time** — the timescale on which two infinitesimally close trajectories drift apart by a factor of $e \approx 2.72$. After about *one* Lyapunov time you've lost a factor-of-$e$ in precision; after *ten* Lyapunov times, your initial-condition error has been amplified by $e^{10} \approx 22{,}000$, and any prediction is meaningless no matter how perfect your model.

For the canonical Lorenz parameters, $\lambda_1 \approx 0.9$, so the **Lyapunov time is $1/\lambda_1 \approx 1.1$ time units**. That's the natural prediction horizon for *any* method — perfect physics simulator or trained ML model — given finite-precision initial conditions. Beyond it, the trajectory could be anywhere on the attractor consistent with the past.

This is the *interesting* case for ML: short-term prediction is possible, long-term is not — and we can see the limit explicitly.

In [ ]:
def lorenz(state: np.ndarray, t: float,
           sigma: float = 10.0, rho: float = 28.0, beta: float = 8/3) -> list[float]:
    """Right-hand side of the Lorenz '63 system."""
    x, y, z = state
    return [sigma * (y - x), x * (rho - z) - y, x * y - beta * z]

dt = 0.01
t  = np.arange(0, 60.0, dt)
traj = odeint(lorenz, [1.0, 1.0, 1.0], t)
x_series = traj[:, 0]                 # we predict the x component only

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(traj[:, 0], traj[:, 2], lw=0.4)
axes[0].set_xlabel('x'); axes[0].set_ylabel('z'); axes[0].set_title('Attractor (x–z projection)')
axes[1].plot(t, x_series, lw=0.5)
axes[1].set_xlabel('time'); axes[1].set_ylabel('x(t)'); axes[1].set_title('x component — what we predict')
fig.tight_layout()

### Visualising the Lyapunov time

Let's *measure* the Lyapunov time directly. Integrate two Lorenz trajectories starting from initial conditions that differ by only $10^{-8}$ in $x$. Plot the distance between them as a function of time — on a log scale, exponential growth becomes a straight line whose slope is $\lambda_1$.

In [ ]:
# Two trajectories starting 1e-8 apart on the attractor.
# (We integrate first for 50 time units to land on the attractor,
# then start the comparison so we measure the *attractor's* Lyapunov exponent,
# not the slower transient dynamics during spin-up.)
t_warm = np.arange(0, 50.0, 0.01)
spin = odeint(lorenz, [1.0, 1.0, 1.0], t_warm, rtol=1e-12, atol=1e-14)
ic_on_attractor = spin[-1]

t_lyap = np.arange(0, 15.0, 0.01)
traj_A = odeint(lorenz, ic_on_attractor,                       t_lyap, rtol=1e-12, atol=1e-14)
traj_B = odeint(lorenz, ic_on_attractor + np.array([1e-8, 0, 0]), t_lyap, rtol=1e-12, atol=1e-14)

dist = np.linalg.norm(traj_A - traj_B, axis=1)

# Fit to the clean exponential-growth region (before saturation)
mask = (t_lyap > 1) & (t_lyap < 9)
slope, intercept = np.polyfit(t_lyap[mask], np.log(dist[mask]), 1)
print(f"fitted largest Lyapunov exponent  λ₁    = {slope:.2f}")
print(f"corresponding Lyapunov time       1/λ₁  = {1/slope:.2f} time units")

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(t_lyap, dist, color='C0', lw=1.0, label=r'distance $\|\delta(t)\|$ between the two trajectories')
ax.semilogy(t_lyap[mask], np.exp(intercept + slope * t_lyap[mask]),
            color='C3', ls='--', lw=1.5,
            label=fr'fit: $e^{{\lambda_1 t}}$ with $\lambda_1\!\approx\!{slope:.2f}$')
ax.axvline(1/slope, color='C2', ls=':', lw=1.0, label=fr'one Lyapunov time $\approx {1/slope:.1f}$')
ax.set_xlabel('time'); ax.set_ylabel('distance between trajectories')
ax.set_title(r'Two Lorenz trajectories starting only $10^{-8}$ apart')
ax.legend(loc='lower right'); fig.tight_layout()

In [ ]:
WINDOW = 30                               # 0.3 time units of recent history
X, y = make_windows(x_series, WINDOW)
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

rf  = RandomForestRegressor(n_estimators=300, max_depth=15,
                           random_state=0, n_jobs=-1).fit(X_train_s, y_train)
mlp = MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=4000,
                  random_state=0).fit(X_train_s, y_train)

for name, m in [('Random Forest', rf), ('Multi-Layer Perceptron', mlp)]:
    yhat = m.predict(X_test_s)
    print(f"{name:25s} one-step RMSE = {np.sqrt(mean_squared_error(y_test, yhat)):.4f}"
          f"   R² = {r2_score(y_test, yhat):.4f}")

In [ ]:
# Free rollout — watch the predictions decorrelate from truth on the Lyapunov timescale
n_future = min(1500, len(x_series) - (WINDOW + split))   # clip to available truth
seed = x_series[split:split + WINDOW]
roll_rf  = rollout(rf,  seed, n_future, scaler=scaler)
roll_mlp = rollout(mlp, seed, n_future, scaler=scaler)

t_test = t[WINDOW + split : WINDOW + split + n_future]
truth  = x_series[WINDOW + split : WINDOW + split + n_future]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(t_test, truth,    color='k',  lw=1.0, label='truth')
ax.plot(t_test, roll_rf,  color='C2', lw=0.9, label='Random Forest')
ax.plot(t_test, roll_mlp, color='C1', lw=0.9, label='Multi-Layer Perceptron')
ax.axvline(t_test[0] + 1.0, color='C3', ls='--', lw=0.8, label=r'$1/\lambda_1 \approx 1$')
ax.set_xlabel('time'); ax.set_ylabel('x(t)')
ax.set_title('Lorenz: free rollout decorrelates after ~1 Lyapunov time')
ax.legend(); fig.tight_layout()

In [ ]:
# Quantify it: per-step error vs. prediction horizon
err_rf  = np.abs(truth - roll_rf)
err_mlp = np.abs(truth - roll_mlp)

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(t_test - t_test[0], err_rf,  color='C2', lw=0.9, label='Random Forest')
ax.semilogy(t_test - t_test[0], err_mlp, color='C1', lw=0.9, label='Multi-Layer Perceptron')
ax.set_xlabel('prediction horizon (time units)')
ax.set_ylabel(r'$|x_{\rm pred} - x_{\rm truth}|$')
ax.set_title('Error growth on the Lorenz attractor')
ax.legend(); fig.tight_layout()

**Discussion.** Both models track the truth for roughly **one Lyapunov time** ($\sim$1 time unit), exactly as predicted by the divergence rate we measured above. After that they drift away from the truth — but they continue to produce *plausible* trajectories that stay on the attractor. This is the signature of a model that has learned the *geometry* of the dynamics without being able to overcome the fundamental sensitivity to initial conditions. No amount of training data fixes this; chaos is a property of the system, not the model. Even a perfect physics simulator, given the same finite-precision starting state, would lose its forecast on the same timescale.

---
## Takeaways

| System         | One-step RMSE | Long rollout                                        | Bottleneck                                 |
|----------------|--------------|------------------------------------------------------|--------------------------------------------|
| Pendulum       | tiny for both models | Linear Regression tracks; Multi-Layer Perceptron diverges | model-class mismatch with linear dynamics  |
| Sunspots       | moderate     | flattens to the long-term average                    | physics not in the features                |
| Lorenz         | tiny         | useful for ~1 Lyapunov time                          | sensitive dependence on initial conditions |

**Three rules of thumb:**
1. Always split a time series **chronologically**, never by random shuffling — random splits leak the future into training.
2. **One-step** accuracy and **rollout** accuracy answer different questions. Report both. A model can have excellent one-step error and still blow up under autoregressive rollout (see the pendulum Multi-Layer Perceptron).
3. ML can learn the **shape** of a system's evolution, but cannot manufacture predictability that the physics does not provide.

## Where to go next
- Recurrent networks (Long Short-Term Memory, Gated Recurrent Unit) and Transformers handle longer-range dependencies than a fixed lag window.
- **Reservoir computing** (echo-state networks) is famously good on Lorenz-type systems with very modest training.
- **Hybrid physics-informed models**: combine a known PDE with a neural correction term — keeps the long-term constraints physics gives you.

## Resources
- Brunton & Kutz, *Data-Driven Science and Engineering* (2019), Ch. 6–7.
- SILSO sunspot data: https://www.sidc.be/SILSO/datafiles
- Pathak et al. (2018), *Model-Free Prediction of Large Spatiotemporally Chaotic Systems from Data*, PRL 120, 024102.